# Phase 6 — Comparaison avec le modèle naïf

## Objectifs

- Créer un modèle de référence qui prédit toujours « pas canular ».
- Évaluer ce modèle sur le même jeu de test que le modèle sans fuite.
- Comparer son accuracy, sa precision et son recall avec le vrai modèle.
- Montrer pourquoi l'accuracy seule est insuffisante pour ce problème.

## Imports

In [3]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## Chemins et colonnes

In [4]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## Recharger les lignes correctes

In [5]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes chargées : {len(df)}")
print(f"Lignes isolées : {len(lignes_problemes)}")

Lignes chargées : 88679
Lignes isolées : 196


## Convertir les types

In [6]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

##  Recréer la cible `is_hoax`

In [7]:
MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

pattern_canular = "|".join(
    re.escape(mot) for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

print(df["is_hoax"].value_counts())
print(df["is_hoax"].value_counts(normalize=True).mul(100).round(2))

is_hoax
0    87810
1      869
Name: count, dtype: int64
is_hoax
0    99.02
1     0.98
Name: proportion, dtype: float64


## Créer les variables utilisables sans fuite

In [8]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["text_features_without_leakage"] = (
    "city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

##  Définir X et y

In [9]:
features_numeriques_sans_fuite = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
]

features_modele_sans_fuite = [
    "text_features_without_leakage",
] + features_numeriques_sans_fuite

X = df[features_modele_sans_fuite].copy()
y = df["is_hoax"].copy()

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (88679, 7)
Dimensions de y : (88679,)


## Recréer le même train/test

In [10]:
indices_train, indices_test = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train = X.loc[indices_train]
X_test = X.loc[indices_test]

y_train = y.loc[indices_train]
y_test = y.loc[indices_test]

print("Taille entraînement :", X_train.shape)
print("Taille test :", X_test.shape)

print(f"Part de canulars dans train : {y_train.mean():.2%}")
print(f"Part de canulars dans test : {y_test.mean():.2%}")

Taille entraînement : (70943, 7)
Taille test : (17736, 7)
Part de canulars dans train : 0.98%
Part de canulars dans test : 0.98%


## Construire et entraîner le modèle sans fuite

In [12]:
preprocessing_sans_fuite = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features_without_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            features_numeriques_sans_fuite,
        ),
    ]
)

modele_sans_fuite = Pipeline(
    steps=[
        ("preprocessing", preprocessing_sans_fuite),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

modele_sans_fuite.fit(X_train, y_train)

y_pred_modele = modele_sans_fuite.predict(X_test)

print("Modèle sans fuite entraîné.")

Modèle sans fuite entraîné.


## Métriques du modèle sans fuite

In [13]:
precision_modele = precision_score(
    y_test,
    y_pred_modele,
    zero_division=0,
)

recall_modele = recall_score(
    y_test,
    y_pred_modele,
    zero_division=0,
)

accuracy_modele = accuracy_score(
    y_test,
    y_pred_modele,
)

print(f"Accuracy du modèle : {accuracy_modele:.2%}")
print(f"Precision du modèle : {precision_modele:.2%}")
print(f"Recall du modèle : {recall_modele:.2%}")

Accuracy du modèle : 75.13%
Precision du modèle : 1.69%
Recall du modèle : 42.53%


## Construire le modèle du stagiaire

In [14]:
modele_stagiaire = DummyClassifier(
    strategy="constant",
    constant=0,
)

modele_stagiaire.fit(X_train, y_train)

y_pred_stagiaire = modele_stagiaire.predict(X_test)

pd.Series(y_pred_stagiaire).value_counts()

0    17736
Name: count, dtype: int64

## Métriques du stagiaire

In [15]:
accuracy_stagiaire = accuracy_score(
    y_test,
    y_pred_stagiaire,
)

precision_stagiaire = precision_score(
    y_test,
    y_pred_stagiaire,
    zero_division=0,
)

recall_stagiaire = recall_score(
    y_test,
    y_pred_stagiaire,
    zero_division=0,
)

print(f"Accuracy du stagiaire : {accuracy_stagiaire:.2%}")
print(f"Precision du stagiaire : {precision_stagiaire:.2%}")
print(f"Recall du stagiaire : {recall_stagiaire:.2%}")

Accuracy du stagiaire : 99.02%
Precision du stagiaire : 0.00%
Recall du stagiaire : 0.00%


## Matrices de confusion

In [16]:
matrice_modele = confusion_matrix(
    y_test,
    y_pred_modele,
)

matrice_stagiaire = confusion_matrix(
    y_test,
    y_pred_stagiaire,
)

df_matrice_modele = pd.DataFrame(
    matrice_modele,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

df_matrice_stagiaire = pd.DataFrame(
    matrice_stagiaire,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

print("Matrice du modèle sans fuite")
display(df_matrice_modele)

print("Matrice du stagiaire")
display(df_matrice_stagiaire)

Matrice du modèle sans fuite


,Prédit : non-canular,Prédit : canular
Réel : non-canular,13251,4311
Réel : canular,100,74


Matrice du stagiaire


,Prédit : non-canular,Prédit : canular
Réel : non-canular,17562,0
Réel : canular,174,0


## Tableau comparatif

In [17]:
resultats_phase6 = pd.DataFrame(
    [
        {
            "modele": "Modèle sans fuite",
            "accuracy": accuracy_modele,
            "precision_canular": precision_modele,
            "recall_canular": recall_modele,
        },
        {
            "modele": "Stagiaire : toujours non-canular",
            "accuracy": accuracy_stagiaire,
            "precision_canular": precision_stagiaire,
            "recall_canular": recall_stagiaire,
        },
    ]
)

resultats_phase6

,modele,accuracy,precision_canular,recall_canular
0,Modèle sans fuite,0.751297,0.016876,0.425287
1,Stagiaire : toujours non-canular,0.990189,0.000000,0.000000


## Exporter les résultats

In [18]:
resultats_phase6.to_csv(
    OUTPUT_DIR / "resultats_phase6_comparaison_stagiaire.csv",
    index=False,
)

df_matrice_modele.to_csv(
    OUTPUT_DIR / "matrice_confusion_modele_sans_fuite.csv",
    index=True,
)

df_matrice_stagiaire.to_csv(
    OUTPUT_DIR / "matrice_confusion_modele_stagiaire.csv",
    index=True,
)

print("Résultats exportés dans outputs/.")

Résultats exportés dans outputs/.
